# Proyecto 1 — Monitoreo transaccional

**Curso:** Deep Learning 2026  
**Estado:** etapas 0 y 1 — diseño previo a cualquier resultado  
**Integrantes:** _completar nombres y apellidos_

> Este notebook no genera datos ni entrena modelos. Primero dejamos congelado qué queremos comparar y qué puede ver cada modelo.

## 1. Pregunta y ruta de datos

La pregunta es: **¿el orden cronológico de las transacciones aporta información adicional para detectar fraude que no pueda capturarse únicamente mediante variables agregadas?**

Usaremos la **Ruta A: datos sintéticos con generador propio**. Esto nos permitirá crear mecanismos cuyo orden importe y otros que puedan detectarse con agregados. Así no diseñamos todo a favor de la GRU. El generador se implementará en la etapa 2; todavía no existe un dataset.

In [ ]:
from src.config import RANDOM_SEED

RANDOM_SEED

La semilla central será `42`. En las siguientes etapas se aplicará a Python `random`, NumPy, PyTorch, el generador y los cargadores. El generador deberá producir exactamente los mismos datos con la misma semilla, parámetros y versión del código.

## 2. Qué representa una predicción

Cada ejemplo corresponde a una **transacción objetivo de una tarjeta**. Simulamos la decisión que ocurre cuando esa operación intenta realizarse. Por eso el sistema puede ver el monto, canal, comercio, hora y demás campos presentes en la solicitud actual, además del historial estrictamente anterior de la tarjeta. No puede ver el resultado futuro de la operación, `is_fraud`, `fraud_type` ni transacciones posteriores.

Todos los modelos devolverán `risk_score ∈ [0, 1]`. No aplicamos todavía un threshold.

- **Modelo secuencial:** 12 eventos anteriores como candidato inicial y la operación objetivo como paso actual. Padding y máscara representarán historiales cortos.
- **Modelo agregado:** operación actual y estadísticas calculadas solo con el historial permitido.

Doce eventos es un comienzo manejable para representar ráfagas cortas, no una elección final. Solo TRAIN y VALIDATION podrán justificar cambiarlo.

## 3. Split temporal, TEST cerrado y leakage

La partición inicial será global y cronológica: primer 70 % para TRAIN, siguiente 15 % para VALIDATION y último 15 % para TEST. No habrá shuffle. Los bloques con el mismo timestamp no se partirán entre conjuntos y más adelante se comprobará:

```python
assert train.timestamp.max() < validation.timestamp.min()
assert validation.timestamp.max() < test.timestamp.min()
```

El historial causal de un objetivo puede cruzar hacia atrás una frontera, porque en producción ese pasado sí estaría disponible. La pertenencia al split la determina la transacción objetivo.

**TEST se usará una sola vez al final.** No servirá para variables, escaladores, encoders, longitud, arquitectura, hiperparámetros, elección de C, threshold ni para decidir si una hipótesis funciona.

Toda transformación aprendida hará `fit` exclusivamente con TRAIN y luego `transform` sobre cada split. Los agregados serán causales; no se usarán estadísticas calculadas al final de la vida de una tarjeta ni información futura.

## 4. Modelos que compararemos

### A — baseline sin orden

Candidato inicial: `HistGradientBoostingClassifier`. Recibirá monto y campos actuales, promedio/desviación/máximo histórico reciente, conteos y frecuencia, diversidad de comercios y canales, tiempo desde la operación anterior y razón entre monto actual y promedio. No recibirá posiciones, listas ordenadas ni transiciones completas. Será un baseline competitivo: usar resúmenes temporales causales no equivale a reconstruir el orden.

### B — GRU

`eventos ordenados → representación numérica → GRU → capa densa → sigmoid → risk_score`

La GRU es el punto de partida por su capacidad secuencial, número moderado de parámetros y claridad. Consideramos RNN simple, LSTM, CNN temporal y Transformer. No afirmamos que GRU sea siempre mejor.

### C — híbrido

`GRU(secuencia) + red densa(agregados) → concatenación → capas densas → risk_score`

**Hipótesis previa:** la secuencia puede representar transiciones y orden, mientras los agregados resumen el comportamiento reciente; combinarlos podría aportar información complementaria.

**Criterio previo:** C será candidato útil si en VALIDATION supera el AUC-PR de B y su costo económico, usando thresholds elegidos con la misma regla en VALIDATION, no es mayor. Si la diferencia es pequeña, un bootstrap agrupado por tarjeta deberá respaldar que no es solo ruido. No se definirá el veredicto mirando TEST.

## 5. Evaluación

La métrica principal será **AUC-PR**, porque el fraude estará desbalanceado. En el threshold elegido después con VALIDATION reportaremos precision, recall y F1. Accuracy será, como máximo, una referencia secundaria.

El costo será:

`economic_cost = false_negatives × Q4,200 + false_positives × Q180`

El threshold se minimizará con VALIDATION, se congelará y solo entonces se aplicará a TEST. Hoy no existe ningún threshold seleccionado. También reportaremos costo por transacción para poder comparar splits de distinto tamaño.

## 6. Pruebas que podrían refutar nuestra explicación

1. **Permutación controlada:** barajaremos los eventos dentro de cada secuencia conservando valores, longitud, agregados y etiqueta. Compararemos B con orden original frente a varias permutaciones reproducibles. Si no hay caída, no diremos que B utilizó el orden.
2. **Recorte de historia:** compararemos 12 eventos anteriores contra 3 sobre el mismo universo de ejemplos. Antes de ejecutarlo predecimos que los fraudes que dependen de una cadena de acciones serán más difíciles con poco contexto.

Ambas pruebas quedan declaradas antes del entrenamiento y se desarrollarán primero con TRAIN/VALIDATION.

## 7. Datos sintéticos que diseñaremos en la etapa 2

Campos iniciales: `timestamp`, `card_id`, `amount`, `merchant_category`, `channel`, `hour`, `day_of_week`, `time_since_previous` y `distance_from_previous`. `is_fraud` será la etiqueta y `fraud_type` se reservará para auditoría; nunca serán predictores.

- **Card testing → cashout:** varias operaciones pequeñas cercanas seguidas por una grande. Reordenar los mismos eventos no debería representar necesariamente el patrón.
- **Cambio anormal de canal/comportamiento:** por ejemplo, ONLINE desconocido → ATM en poco tiempo respecto a un perfil normalmente POS.
- **Anomalía de monto:** gasto extraordinario respecto a una historia estable; será detectable razonablemente mediante agregados.

También habrá vuelos, vacaciones, gastos extraordinarios legítimos, cambios válidos de canal, ráfagas de compras y usuarios irregulares. Esperamos que un viaje legítimo con canal nuevo, distancia alta y monto grande sea un caso difícil. Auditaremos que canal, periodo, comercio o rango de monto no delaten por sí solos la etiqueta.

## 8. Justicia experimental y control de calidad

A, B y C usarán las mismas transacciones objetivo, etiqueta, horizonte, fronteras y TEST. Los historiales cortos no se descartarán solo para un modelo. B no tendrá futuro que A no tenga y C no recibirá columnas auxiliares.

- [x] No hay split aleatorio ni preprocessing ajustado con todos los datos.
- [x] TEST no se utilizó.
- [x] No se eligió arquitectura con TEST ni threshold.
- [x] No se entrenaron modelos ni se inventaron métricas o resultados.
- [x] No se afirmó que el orden aporta.
- [x] La hipótesis de C y las falsificaciones quedaron escritas previamente.
- [x] La comparación tendrá el mismo universo y horizonte.

## 9. Próximo paso (no ejecutado)

En la etapa 2 se especificará e implementará el generador reproducible, se validarán causalidad, prevalencia y ausencia de atajos, y recién entonces se crearán datos versionados. No avanzamos hacia esa etapa en este notebook.